

Denne skabelon indeholder fire sektioner: **Opgave 1** til **Opgave 4**. Hver sektion har fire underspørgsmål: **Spørgsmål 1** til **Spørgsmål 4**.
En notebook konverteres til pdf ved at køre: jupyter nbconvert --to pdf ExamTemplate2026.ipynb i terminalen.
Når notebooken eksporteres/printes til PDF, starter hver ny opgave på en ny side.


<style>
@media print {
  .pagebreak { page-break-before: always; break-before: page; }
}
</style>


## Kopiérbar blok til indsættelse af billeder

### Markdown-metode
Kopiér linjen nedenfor til en Markdown-celle og ret filnavn/størrelse efter behov:

```markdown
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
```

### Python-metode
Kør kodecellen nedenfor, og brug funktionen `indsæt_billede(...)` i dine svar.


In [1]:
from IPython.display import Image, display

def indsæt_billede(sti, bredde=600):
    """Viser et billede i notebooken.

    Parametre:
    sti: Filsti til billedet, fx 'images/figur1.png'
    bredde: Billedets bredde i pixels, fx 600
    """
    display(Image(filename=sti, width=bredde))

# indsæt_billede('tree.png', bredde=600)


In [2]:
# Imports
import numpy as np
import pulp as PLP

# Opgave 1

## Spørgsmål 1

The branch and bound method is used to solve integer linear programming problems (ILP) by iteratively restricting the feasible region of a linear programming relaxation of the problem. The method involves branching on decision variables, creating subproblems, and using bounds to eliminate subproblems that cannot yield better solutions than the best known solution. The process continues until all subproblems have been explored or eliminated, resulting in the optimal integer solution.

Soltuion are presented such that $value^*$ is a feasible solution, variables values are in (x_1, x_2) and

To solve the intermediate problems i use pulp. I constuct the model below

In [3]:
x_range = range(2)
x = PLP.LpVariable.dicts("x", indices=x_range, cat = PLP.LpContinuous, lowBound=0, upBound=None)
model = PLP.LpProblem("ILP", sense=PLP.LpMaximize)

#Objective
obj_coef = [3,2]
model += PLP.lpSum(obj_coef[i]*x[i] for i in x_range)

#Constraint 1
c1_coef = [4,2]
model += PLP.lpSum(c1_coef[i]*x[i] for i in x_range) <= 15

# Constraint 2
c2_coef = [2,3]
model += PLP.lpSum(c2_coef[i]*x[i] for i in x_range) <= 12


I then implemented a code, that uses pulp and a recursive procedure that solves the branch and bound problem

In [4]:
import pulp as PLP
import numpy as np
from math import ceil, floor
import copy

def branch_and_bound(LPmodel, sense, best_so_far = [None], objectives = [None], problem = [0]):

    """
    :param LPmodel:
    input a Linear Programming relaxation of the ILP problem.
    Note it is import to keep track of if it is a minimization or maximization problem, as this will determine the
    branching strategy and pruning strategy.
    :return:
    Iterative Branch and Bound solution to the ILP problem.
    """
    if best_so_far[0] is None:
        if sense == "maximize":
            best_so_far[0] = -10 ** 6
        else:
            best_so_far[0] = 10 ** 6

    if sense != "maximize" and sense != "minimize":
        raise ValueError("sense must be either 'maximize' or 'minimize'")
    if sense == "maximize":
        def objective_is_not_better(obj):
            return obj < best_so_far[0]
    if sense == "minimize":
        def objective_is_not_better(obj):
            return obj > best_so_far[0]


    def is_integer_value():
        """
        :return:
        dict where keys are variable names and values are boolean values indicating whether variable is integer.
        """
        eps = 10**-4
        vars = LPmodel.variables()

        d = dict()
        # Assign boolean values to the decision variables based on whether they are integer or not
        for var in vars:
            if abs(var.varValue - ceil(var.varValue)) > eps and abs(var.varValue - floor(var.varValue)) > eps:
                d[var.name] = False
            else:
                d[var.name] = True
        return d
    print(20*"#")
    print("Problem : ", problem[0])
    print(20 * "#")
    print()
    ### Step 1 - Solve problem ###
    LPmodel.solve(PLP.PULP_CBC_CMD(msg = 0))
    obj = PLP.value(LPmodel.objective)
    ### Step 2 - Branch on non-integer decision variable
    vars = is_integer_value()
    if LPmodel.status == PLP.LpStatusInfeasible:
        print("Pruning branch, infeasible\n")
        return objectives
    if objective_is_not_better(obj):
        print("Pruning branch with objective value ", obj, " which is worse than best so far ", best_so_far[0])
        print("Decision variables: ")
        for v in LPmodel.variables():
            print(v.name, "=", v.varValue)
        print("Objective value: ", obj)
        print()
        return objectives
    if all([v for v in vars.values()]):
        print("Found feasible branch, backtracking")
        print("Decision variables: ")
        for v in LPmodel.variables():
            print(v.name, "=", v.varValue)
        print("Objective value: ", obj)
        print()
        if obj > best_so_far[0] and sense == "maximize":
            best_so_far[0] = obj
        if obj < best_so_far[0] and sense == "minimize":
            best_so_far[0] = obj
        objectives[0] = obj
        return objectives


    for name, value in vars.items():
        if value:
            continue
        else:
            branch_name = name
            branch_value = LPmodel.variablesDict()[branch_name].varValue
            break
    ### Step 3 - Create two branches and solve recursively ###
    ### Base case - All decision variables or the problem is not feasible or objective does not become better ###


    # Left branch
    left_model = copy.deepcopy(LPmodel)
    left_model += left_model.variablesDict()[branch_name] <= floor(branch_value)

    for v in LPmodel.variables():
        print(v.name, "=", v.varValue)
    print("Objective: ", obj)
    print("Adding constraint ", branch_name, " <= ", floor(branch_value), " to left branch")
    print()
    problem[0] = problem[0] + 1
    branch_and_bound(left_model, sense, best_so_far, objectives, problem)

    # Right branch
    right_model = copy.deepcopy(LPmodel)
    right_model += right_model.variablesDict()[branch_name] >= ceil(branch_value)
    print("Adding constraint ", branch_name, " >= ", ceil(branch_value), " to right branch")
    print()
    problem[0] = problem[0] + 1
    branch_and_bound(right_model, sense, best_so_far, objectives, problem)

    return best_so_far[0]

I use the implemented function to solve the branch and bound problem, using the defined model.

In [5]:
branch_and_bound(model, sense = "maximize")

####################
Problem :  0
####################

x_0 = 2.625
x_1 = 2.25
Objective:  12.375
Adding constraint  x_0  <=  2  to left branch

####################
Problem :  1
####################

x_0 = 2.0
x_1 = 2.6666667
Objective:  11.3333334
Adding constraint  x_1  <=  2  to left branch

####################
Problem :  2
####################

Found feasible branch, backtracking
Decision variables: 
x_0 = 2.0
x_1 = 2.0
Objective value:  10.0

Adding constraint  x_1  >=  3  to right branch

####################
Problem :  3
####################

x_0 = 1.5
x_1 = 3.0
Objective:  10.5
Adding constraint  x_0  <=  1  to left branch

####################
Problem :  4
####################

Pruning branch with objective value  9.6666666  which is worse than best so far  10.0
Decision variables: 
x_0 = 1.0
x_1 = 3.3333333
Objective value:  9.6666666

Adding constraint  x_0  >=  2  to right branch

####################
Problem :  5
####################

Pruning branch, infeasible

Adding c

11.0

We reach the optimal solution of $11$ with $x_1 = 3$ and $x_2 = 1$, the tree is displayed below.

## Spørgsmål 2

We now solve the problem as an ILP using pulp, to do so i introduce integer contraints on the variables


In [6]:
x_range = range(2)
x = PLP.LpVariable.dicts("x", indices=x_range, cat = PLP.LpInteger, lowBound=0, upBound=None)
model = PLP.LpProblem("ILP", sense=PLP.LpMaximize)

#Objective
obj_coef = [3,2]
model += PLP.lpSum(obj_coef[i]*x[i] for i in x_range)

#Constraint 1
c1_coef = [4,2]
model += PLP.lpSum(c1_coef[i]*x[i] for i in x_range) <= 15

# Constraint 2
c2_coef = [2,3]
model += PLP.lpSum(c2_coef[i]*x[i] for i in x_range) <= 12

model.solve()
print(PLP.LpStatus[model.status])
print("Obj:", PLP.value(model.objective))
for v in model.variables():
    print(v.name, "=", v.varValue)


Optimal
Obj: 11.0
x_0 = 3.0
x_1 = 1.0


\newpage

# Opgave 2

This is a critical path problem, we can use a network model to solve this. I have implemented a class below for the critical path problem, this will lay the basis for the solutions i the next questions.


In [7]:
import pulp as PLP

class critical_path_problem:
    """
    This class defines the critical path problem. The implementation follows from uge 7 Afsnit 5.3 Netværksmodeller
    slide 42.

    Input:
    Nodes - list of integers
    Edges - list of integers
    times - dict of time associated with each edge

    """

    def __init__(self,nodes,edges, times):
        self.nodes = nodes
        self.edges = edges
        self.times = times


        #Model, decision variable and objective
        self.model = PLP.LpProblem("critical_path_problem", sense=PLP.LpMinimize)

        self.z = PLP.LpVariable("z", lowBound=0, cat=PLP.LpContinuous)
        self.t = PLP.LpVariable.dicts("t", range(len(self.nodes)), lowBound=0, cat=PLP.LpContinuous)

        self.model += self.z, "Objective"

    def construct_constraints(self):
        for i,j in self.edges:
            # Ending j must be greater than start i + time for activity i to j
            self.model += self.t[j] >= self.t[i] + self.times[(i,j)]
        for j in self.nodes:
            # decision variable t must be less than or equal to z for all nodes
            self.model += self.t[j] <= self.z

        self.constraints = "ADDED"

    def solve_and_print(self):
        if self.constraints != "ADDED":
            raise Exception("You must add constraints before solving")

        self.model.solve()

        print("Status:", PLP.LpStatus[self.model.status])
        print("Objective value:", PLP.value(self.model.objective))

        for j in self.nodes:
            print(f"Node {j} has time {round(self.t[j].varValue, 2)}")
        for var in self.model.variables():
            if "t" in var.name:
                print(var.name, ":", var.value())


## Spørgsmål 1

Explaination for the contraint may be seen as code comments, i solve the LP problem from figure 1. The critical path is to bee understood such that this is the sequence of event that takes the minimal total time, under the contraint that events of the nodes with outgoing edges must be completed before we can start the event at the incoming nodes.

In [9]:
nodes = list(range(5))

# (i,j,cost)         A         B        C        D        E        F         G
edges_and_cost = [(0,1,8), (0,2,14), (1,2,8), (1,3,7), (2,3,8), (2,4,14), (3,4,7)]
edges = []
cost = {}
for i, j , c in edges_and_cost:
    e = (i,j)
    edges.append(e)
    cost[e] = c
CPP = critical_path_problem(nodes,edges,cost)
CPP.construct_constraints()
CPP.solve_and_print()


Status: Optimal
Objective value: 31.0
Node 0 has time 0.0
Node 1 has time 8.0
Node 2 has time 16.0
Node 3 has time 24.0
Node 4 has time 31.0
t_0 : 0.0
t_1 : 8.0
t_2 : 16.0
t_3 : 24.0
t_4 : 31.0


## Spørgsmål 2

We wish to implement a solution such that it is reflected that time is reduced, at the cost of speeding up certain processes. We need to add indicator variables reflecting if a edge has a reduces cost. We cannot surpass the budget and we can only reduce times by either 3 or 5 once in total.

In [19]:
CCP_speed_up = critical_path_problem(nodes,edges,cost)
# Indicator
delta3 = PLP.LpVariable.dicts("delta3", indices= edges, cat = PLP.LpBinary)
delta5 = PLP.LpVariable.dicts("delta5", indices= edges, cat = PLP.LpBinary)
price_3 = 2000
price_5 = 4000
budget = 10000

# Budget constraint
CCP_speed_up.model += PLP.lpSum(delta3[e] * price_3 + delta5[e] * price_5
                                for e in edges) <= budget
# Maximum one reduction per edge
for e in edges:
    CCP_speed_up.model += delta3[e] + delta5[e] <= 1
for i,j in CCP_speed_up.edges:
            # Ending j must be greater than start i + time for activity i to j
            CCP_speed_up.model += CCP_speed_up.t[j] >= CCP_speed_up.t[i] + CCP_speed_up.times[(i,j)] - delta3[(i,j)]*3 - delta5[(i,j)]*5
for j in nodes:
    CCP_speed_up.model += CCP_speed_up.t[j] <= CCP_speed_up.z

CCP_speed_up.constraints = "ADDED"
CCP_speed_up.solve_and_print()

Status: Optimal
Objective value: 23.0
Node 0 has time 0.0
Node 1 has time 5.0
Node 2 has time 11.0
Node 3 has time 16.0
Node 4 has time 23.0
delta3_(0,_1) : 1.0
delta3_(0,_2) : 1.0
delta3_(1,_2) : 1.0
delta3_(1,_3) : 0.0
delta3_(2,_3) : 1.0
delta3_(2,_4) : 1.0
delta3_(3,_4) : 0.0
delta5_(0,_1) : 0.0
delta5_(0,_2) : 0.0
delta5_(1,_2) : 0.0
delta5_(1,_3) : 0.0
delta5_(2,_3) : 0.0
delta5_(2,_4) : 0.0
delta5_(3,_4) : 0.0
t_0 : 0.0
t_1 : 5.0
t_2 : 11.0
t_3 : 16.0
t_4 : 23.0


## Spørgsmål 3

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
############################## Spm. 1) #######################################

"""
I denne opgave ønsker vi at placere gates således at det vægtede flow
minimeres. Da der ingen interaktion er mellem flyene kan dette modelleres som
et normal assignment problem.
"""
gates = "A B C D E".split()
distances = [150, 200, 250, 400, 500]
flows = [60, 50, 20 , 90, 40]
n = 5
cost_matrix = np.zeros((n,n))

for i in range(n):
    for j in range(n):
        cost_matrix[i][j] = distances[j] * flows[i]
print("Cost matrix:", cost_matrix)

from modeller.network.assignment_problem import AssignmentProblem
variable_dict = dict((i,gates[j]) for i in range(n) for j in range(n))
assignment_problem = AssignmentProblem(n,
                                       cost_matrix = cost_matrix,
                                       name ="GateAssignment")

# Rename according to gate names.
for i in assignment_problem.variable_range:
    for j in assignment_problem.variable_range:
        assignment_problem.x[i][j].name = f"x_{i}_{gates[j]}"
assignment_problem.solve()

################################ Spm. 2) ######################################

"""
Vi betragter nu en situation hvor kun kigger på inter-transit flow,
og ønsker at minimere det vægtede flow mellem gates. Dette er et QAP problem.
"""
from modeller.heltalsmodeller.quadtratic_assignment_problem import QuadraticAssignmentProblem

distances = np.array([[0, 150, 200, 250, 400, 500],
                      [150, 0, 50, 100, 250, 350],
                      [200, 50, 0, 50, 300, 400],
                      [250, 100, 50, 0, 250, 350],
                      [400, 250, 300, 250, 0, 300],
                      [500, 350, 400, 350, 300, 0]])

flows = np.zeros((6, 6))

# upper triangle values
flows[0, 1:] = [60, 50, 20, 90, 40]
flows[1, 2:] = [10, 15, 2, 12]
flows[2, 3:] = [3, 20, 35]
flows[3, 4:] = [8, 11]
flows[4, 5:] = [9]

# make symmetric
flows = flows + flows.T

# Check symmetry
if all(distances[i][j] == distances[j][i] for i in range(n) for j in range(n)):
    print("Symmetric")
else:
    raise ValueError("Non-symmetric Matrix")

distances_no_gate = distances[1:,1:]
flows_no_gate = flows[1:,1:]

machine_range = range(n)
location_range = range(n)
QAD_no_gate = QuadraticAssignmentProblem(machine_range = machine_range,
                                         location_range = location_range,
                                 flow_matrix = flows_no_gate,
                                 distance_matrix = distances_no_gate)


# Rename according to gate names.
for i in QAD_no_gate.machine_range:
    for j in QAD_no_gate.location_range:
        QAD_no_gate.x[i][j].name = f"x_{i}_{gates[j]}"
QAD_no_gate.construct_model()
QAD_no_gate.construct_constraints()
QAD_no_gate.solve()

##################################### Spm. 3) ################################
"""
We now also consider the traffic from the planes to the gates. This constitutes
a mixed AP-QAP problem, where we edit the obejctive in the QAP, such that
the linear AP is also considered.
"""
# Standard QAD
QAD = QuadraticAssignmentProblem(machine_range = machine_range,
                                location_range = location_range,
                                 flow_matrix = flows_no_gate,
                                 distance_matrix= distances_no_gate)
# Update the obejctive
QAD.construct_model()

QAD.model.objective = QAD.model.objective + PLP.lpSum(QAD.x[i][j]*
                                                      cost_matrix[i][j]
                                                      for i in range(n)
                                                      for j in range(n))
QAD.construct_constraints()

# Rename according to gate names.
for i in QAD.machine_range:
    for j in QAD.location_range:
        QAD.x[i][j].name = f"x_{i+1}_{gates[j]}"
print("QAP-AP")
QAD.solve()

print("QAP with dummy plane, forced to index 0")
n = n + 1
machine_range = range(n)
location_range = range(n)
QAD = QuadraticAssignmentProblem(machine_range = machine_range,
                                location_range = location_range,
                                 flow_matrix = flows,
                                 distance_matrix = distances)
# Zeroth index is a dummy plane. We must make the location of this plane fixed.
# Dummy plane is fixed to first gate, so we add the constraint that x[0][0] = 1
QAD.construct_model()
QAD.model += QAD.x[0][0] == 1, "DummyPlaneConstraint"
QAD.construct_constraints()
# Rename according to gate names.
gates = ["IU"] + gates
for i in QAD.machine_range:
    for j in QAD.location_range:
        QAD.x[i][j].name = f"x_{i}_{gates[j]}"
QAD.solve()


## Spørgsmål 4

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
# Beregninger/kode til dette spørgsmål kan skrives her


\newpage

# Opgave 3

Skriv din besvarelse nedenfor.


## Spørgsmål 1

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
# Beregninger/kode til dette spørgsmål kan skrives her


## Spørgsmål 2

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
# Beregninger/kode til dette spørgsmål kan skrives her


## Spørgsmål 3

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
%%sql


In [ ]:
# Beregninger/kode til dette spørgsmål kan skrives her


## Spørgsmål 4

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
# Beregninger/kode til dette spørgsmål kan skrives her


\newpage

# Opgave 4

Skriv din besvarelse nedenfor.


## Spørgsmål 1

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
# Beregninger/kode til dette spørgsmål kan skrives her


## Spørgsmål 2

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
# Beregninger/kode til dette spørgsmål kan skrives her


## Spørgsmål 3

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
# Beregninger/kode til dette spørgsmål kan skrives her


## Spørgsmål 4

_Indsæt din besvarelse her._

<!-- Billede, hvis relevant:
<img src="images/dit_billede.png" alt="Kort beskrivelse" width="600">
-->


In [ ]:
# Beregninger/kode til dette spørgsmål kan skrives her
